# 04 - LLM Agent API Layer

This notebook explains the first LLM-backed layer of the Notely AI PM Analytics Agent.

At this point, the project already has trusted analytics tools. The new question is: how can an LLM decide which tool to call, while Python keeps database access safe and deterministic?

## Mental Model

The LLM is the planner and communicator. The Python tool layer is the executor.

1. PM asks a natural-language question.
2. LLM chooses a tool and structured arguments.
3. Python validates and runs the tool.
4. Tool output is sent back to the LLM.
5. LLM writes a PM-friendly answer.

This is the practical version of function calling.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## 1. Configuration

`agent/config.py` is the place to look when you want to understand how provider, model, API key, and runtime settings are selected.

The default is local Ollama so repeated testing can be free.

In [ ]:
from agent.config import AgentConfig

config = AgentConfig.from_env()
config

To switch providers later, change environment variables before starting Python or Jupyter.

```bash
export NOTELY_LLM_PROVIDER=ollama
export NOTELY_LLM_MODEL=llama3.1:8b
```

For OpenAI later:

```bash
export NOTELY_LLM_PROVIDER=openai
export OPENAI_API_KEY="your_api_key_here"
export NOTELY_LLM_MODEL=gpt-4.1-mini
```

## 2. Tool Schemas

`agent/tool_registry.py` is where existing Python functions become LLM-callable tools. The schema tells the model which function names exist and what arguments are valid.

In [ ]:
from agent.tool_registry import TOOL_SCHEMAS

[(schema['function']['name'], schema['function']['description']) for schema in TOOL_SCHEMAS]

## 3. Tool Execution Is Still Local Python

Even after adding an LLM, the database work still happens through local trusted functions. This is why we built and tested `agent/tools.py` first.

In [ ]:
from agent.tool_registry import execute_tool

execute_tool(
    'get_metric',
    {
        'metric_name': 'activation_rate',
        'start_date': '2026-05-01',
        'end_date': '2026-05-31',
        'group_by': 'platform',
    },
)

## 4. Provider Layer

`agent/providers.py` hides the differences between Ollama and OpenAI. The rest of the agent asks for a chat response and tool calls; the provider decides how that request is sent to the model service.

In [ ]:
from agent.providers import build_provider

provider = build_provider(config)
type(provider).__name__

## 5. Agent Loop

`agent/pm_agent.py` is the loop:

- send user question to the LLM
- receive tool call request
- execute approved tool locally
- send tool result back
- repeat until the model gives a final answer

The trace is important because it lets you check whether the agent's answer matches the actual tool outputs.

In [ ]:
# This cell requires Ollama to be installed and running.
# In a terminal:
#   ollama pull llama3.1:8b
#   ollama serve

from agent.pm_agent import PMAnalyticsAgent

# agent = PMAnalyticsAgent.from_env()
# result = agent.ask('Why did iOS activation change around mid May 2026?')
# print(result.answer)
# result.trace

## 6. Command-Line Demo

The same flow can be tested from the terminal:

```bash
python3 llm_agent_demo.py --check
python3 llm_agent_demo.py --show-trace "Why did iOS activation change around mid May 2026?"
```

Use `--show-trace` when you want to compare the agent answer with the actual tools it called.

## What This Step Adds To The Portfolio Story

This is the first point where the project becomes a real analytics agent instead of only a SQL project.

The important design choice is that the LLM is not trusted with unrestricted database access. It can only call approved tools, and those tools enforce read-only SQL and reusable metric definitions.